# Digital Image Processing Lab - Assignment 7
**Problem Statement:**
Write a program to sharpen the input image using:
1. **Laplacian filter**
2. **Gradient operators of Sobel and Canny edge detector**
3. **Unsharp masking.** *To obtain blurred image, use the Gaussian filter of size 5x5 and sigma= 3*

---
### Mathematical Formulation
Image sharpening highlights fine details and edges by computing spatial derivatives:

1. **Laplacian Filter (2nd Derivative):**
   $$\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}$$
   Discrete kernel with negative center: $\begin{bmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{bmatrix}$.
   Sharpened image:
   $$g(x, y) = f(x, y) - \nabla^2 f(x, y)$$

2. **Sobel Gradient Operator (1st Derivative):**
   $$G_x = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix} * f, \quad G_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix} * f$$
   Magnitude: $G = \sqrt{G_x^2 + G_y^2}$. Sharpened: $g(x, y) = f(x, y) + c \cdot G(x, y)$.

3. **Canny Edge Detector:**
   Multi-stage algorithm: Gaussian smoothing $\to$ Gradient calculation $\to$ Non-maximum suppression $\to$ Hysteresis thresholding.

4. **Unsharp Masking:**
   - Step 1: Blur image $f_{\text{blur}} = f * G_{5\times 5, \sigma=3}$
   - Step 2: Mask $g_{\text{mask}} = f - f_{\text{blur}}$
   - Step 3: Sharpened $g = f + k \cdot g_{\text{mask}}$ (where $k=1$ for unsharp masking)


In [ ]:
# Cell 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'


In [ ]:
# Cell 2: Test Image Generator
def get_blur_edge_image():
    # Slightly blurred test image to demonstrate sharpening
    img = np.zeros((256, 256), dtype=np.float32)
    cv2.circle(img, (128, 128), 70, 220, -1)
    cv2.rectangle(img, (60, 60), (100, 100), 100, -1)
    cv2.putText(img, "SHARP", (65, 140), cv2.FONT_HERSHEY_SIMPLEX, 1.1, 40, 2)
    # Apply slight blur to simulate soft original
    soft = cv2.GaussianBlur(img, (5, 5), 1.0)
    return soft.astype(np.uint8)

test_img = get_blur_edge_image()
print("Test image prepared.")


In [ ]:
# Cell 3: Task 1 - Laplacian Sharpening
def laplacian_sharpen(img):
    # Laplacian kernel with negative center (-4)
    kernel_lap = np.array([[0,  1, 0],
                           [1, -4, 1],
                           [0,  1, 0]], dtype=np.float32)
    # Convolve to get 2nd derivative
    lap = cv2.filter2D(img.astype(np.float32), -1, kernel_lap)
    
    # Sharpened: g = f - lap (since center is negative)
    sharp = img.astype(np.float32) - lap
    sharp = np.clip(sharp, 0, 255).astype(np.uint8)
    
    # Normalize Laplacian for visualization
    lap_display = cv2.normalize(lap, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    return sharp, lap_display

sharp_lap, lap_map = laplacian_sharpen(test_img)
print("Laplacian sharpening complete.")


In [ ]:
# Cell 4: Task 2 - Sobel & Canny Operators
def sobel_sharpen(img, scale=0.5):
    sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(sobelx**2 + sobely**2)
    
    sharp = img.astype(np.float64) + scale * grad_mag
    sharp = np.clip(sharp, 0, 255).astype(np.uint8)
    grad_display = np.clip(grad_mag, 0, 255).astype(np.uint8)
    return sharp, grad_display

def canny_sharpen(img, low_thresh=50, high_thresh=150):
    edges = cv2.Canny(img, low_thresh, high_thresh)
    sharp = np.clip(img.astype(np.int16) + edges.astype(np.int16), 0, 255).astype(np.uint8)
    return sharp, edges

sharp_sobel, sobel_map = sobel_sharpen(test_img, scale=0.4)
sharp_canny, canny_edges = canny_sharpen(test_img, 50, 150)
print("Sobel and Canny sharpening complete.")


In [ ]:
# Cell 5: Task 3 - Unsharp Masking (Gaussian 5x5, sigma=3)
def unsharp_masking(img, k=1.0):
    # Gaussian filter: size 5x5, sigma=3
    blurred = cv2.GaussianBlur(img, (5, 5), sigmaX=3.0, sigmaY=3.0)
    
    # Mask = Original - Blurred
    mask = cv2.subtract(img, blurred)
    
    # Sharpened = Original + k * Mask
    sharp = cv2.addWeighted(img, 1.0, mask, float(k), 0)
    return sharp, blurred, mask

sharp_unsharp, blurred_img, mask_img = unsharp_masking(test_img, k=1.5)
print("Unsharp masking complete.")


In [ ]:
# Cell 6: Visualizations
fig, axes = plt.subplots(3, 3, figsize=(16, 14))

# Row 1: Laplacian
axes[0, 0].imshow(test_img); axes[0, 0].set_title("Original Image", fontweight='bold'); axes[0, 0].axis('off')
axes[0, 1].imshow(lap_map); axes[0, 1].set_title("Laplacian Edge Response", fontweight='bold'); axes[0, 1].axis('off')
axes[0, 2].imshow(sharp_lap); axes[0, 2].set_title("1. Laplacian Sharpened", fontweight='bold'); axes[0, 2].axis('off')

# Row 2: Sobel & Canny
axes[1, 0].imshow(sobel_map); axes[1, 0].set_title("Sobel Gradient Magnitude", fontweight='bold'); axes[1, 0].axis('off')
axes[1, 1].imshow(sharp_sobel); axes[1, 1].set_title("2a. Sobel Sharpened", fontweight='bold'); axes[1, 1].axis('off')
axes[1, 2].imshow(sharp_canny); axes[1, 2].set_title("2b. Canny Edge Enhanced", fontweight='bold'); axes[1, 2].axis('off')

# Row 3: Unsharp Masking
axes[2, 0].imshow(blurred_img); axes[2, 0].set_title("Gaussian Blurred (5x5, sigma=3)", fontweight='bold'); axes[2, 0].axis('off')
axes[2, 1].imshow(mask_img); axes[2, 1].set_title("Unsharp Mask (Original - Blur)", fontweight='bold'); axes[2, 1].axis('off')
axes[2, 2].imshow(sharp_unsharp); axes[2, 2].set_title("3. Unsharp Masked Result", fontweight='bold'); axes[2, 2].axis('off')

plt.tight_layout()
plt.show()


### Comparison & Conclusions
- **Laplacian:** An isotropic 2nd-order derivative filter that emphasizes fine transitions and corners equally in all directions.
- **Sobel & Canny:** 1st-order gradient operators; Sobel provides continuous edge magnitudes while Canny provides thin, localized single-pixel edge boundaries.
- **Unsharp Masking:** Subtracts a smooth Gaussian base to isolate and boost high-frequency details, producing enhanced contrast across borders.
